In [1]:
!pip install -q -U bitsandbytes 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.9 MB/s eta 0:00:00


In [ ]:
# import pandas as pd
# import numpy as np
# from sklearn.model_selection import train_test_split
# from datasets import load_dataset

# # 设定与训练代码完全一致的随机种子
# SEED = 42

# def get_training_samples(dataset_name, samples_per_class=10):
#     print(f"\n{'='*30} 数据集: {dataset_name} (Training Set Only) {'='*30}")
    
#     # 1. 加载原始数据
#     if dataset_name == "SMP2020":
#         ds = load_dataset("Um1neko/smp2020", split="train")
#         df = pd.DataFrame(ds).rename(columns={"content": "text"})
#         label_names = {0: "Angry", 1: "Sad", 2: "Fear", 3: "Neutral", 4: "Happy", 5: "Surprise"}
#     elif dataset_name == "SST-5":
#         ds = load_dataset("SetFit/sst5", split="train")
#         df = pd.DataFrame(ds).rename(columns={"sentence": "text"})
#         label_names = {0: "Very Negative", 1: "Negative", 2: "Neutral", 3: "Positive", 4: "Very Positive"}
#     elif dataset_name == "TweetEval":
#         ds = load_dataset("tweet_eval", "sentiment", split="train")
#         df = pd.DataFrame(ds)
#         label_names = {0: "Negative", 1: "Neutral", 2: "Positive"}
    
#     df = df[["text", "label"]].dropna()
#     df["label"] = df["label"].astype(int)

#     # 2. 严格执行 8:2 切分，只保留训练池 (前 80%)
#     train_pool, _ = train_test_split(
#         df, test_size=0.2, stratify=df["label"], random_state=42 # 必须是42以对齐
#     )

#     # 3. 遍历每个类别输出样本
#     for lbl, name in label_names.items():
#         print(f"\n--- 类别 {lbl} [{name}] 的样本 ---")
#         class_samples = train_pool[train_pool["label"] == lbl]
        
#         # 随机抽取 10 条（或该类所有数据）
#         n_to_show = min(len(class_samples), samples_per_class)
#         display_samples = class_samples.sample(n=n_to_show, random_state=SEED)
        
#         for i, (idx, row) in enumerate(display_samples.iterrows()):
#            
#             clean_text = row['text'].replace('\n', ' ')
#             print(f"{i+1}. [ID:{idx}] {clean_text}")

# # 执行输出
# datasets = ["SMP2020", "SST-5", "TweetEval"]
# for ds in datasets:
#     get_training_samples(ds, samples_per_class=8) 

In [ ]:
import time
import torch
import random
import re
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import train_test_split
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# =========================== ⚙️ 1. 模型配置区域 ===========================
# 选项 A: Qwen2.5-7B-Instruct (先跑这个)
CURRENT_MODEL = "Qwen/Qwen2.5-7B-Instruct" 
# 选项 B: Llama-3.1-8B-Instruct (跑完重启 Kernel 再跑这个)
# CURRENT_MODEL = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit" 

SEED = 42
# ==============================================================================

print(f"🤖 正在初始化模型: {CURRENT_MODEL} ...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(CURRENT_MODEL, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    CURRENT_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
if tokenizer.pad_token_id is None: tokenizer.pad_token_id = tokenizer.eos_token_id

# 获取大致参数量 (Million)
MODEL_PARAMS_M = sum(p.numel() for p in model.parameters()) / 1e6

# --- 2. 标签映射与兜底策略 ---
LABEL_MAPPING = {
    "SMP2020": {"fallback": 3, "max_id": 5},
    "SST-5": {"fallback": 2, "max_id": 4},
    "TweetEval": {"fallback": 1, "max_id": 2}
}

# --- 3. 验证集加载 (100% 复刻小模型的验证集逻辑) ---
def get_validation_set(dataset_name):
    # 强制固定种子，保证与小模型切出的验证集完全一致
    seed_val = 42 
    np.random.seed(seed_val)
    random.seed(seed_val)
    
    if dataset_name == "SMP2020":
        ds = load_dataset("Um1neko/smp2020", split="train")
        df = pd.DataFrame(ds).rename(columns={"content": "text"})
        val_count = 50  # 与 HiPro SMP2020 对齐
    elif dataset_name == "SST-5":
        ds = load_dataset("SetFit/sst5", split="train")
        df = pd.DataFrame(ds).rename(columns={"sentence": "text"})
        val_count = 80  # 与 HiPro SST-5 对齐
    elif dataset_name == "TweetEval":
        ds = load_dataset("tweet_eval", "sentiment", split="train")
        df = pd.DataFrame(ds)
        val_count = 50  # 与 HiPro TweetEval 对齐
        
    df = df[["text", "label"]].dropna()
    df["label"] = df["label"].astype(int)
    
    # 直接拿全量数据按 42 切分。Prompt里的14个例子本身就在train_pool里。
    # 这样切出来的 val_pool 和小模型代码一模一样，且天然杜绝了泄漏！
    _, val_pool = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=seed_val)
    
    num_labels = df['label'].nunique()
    sampled_dfs = []
    for label in range(num_labels):
        class_df = val_pool[val_pool['label'] == label]
        n_samples = min(len(class_df), val_count)
        sampled_dfs.append(class_df.sample(n=n_samples, random_state=seed_val))
    
    val_df = pd.concat(sampled_dfs).sample(frac=1, random_state=seed_val).reset_index(drop=True)
    return val_df

# --- 4. 极致打磨的 CoT Prompt 构造器 ---
def build_cot_prompt(dataset_name, text, method):
    if dataset_name == "SMP2020":
        task_desc = "任务：请通过一步步的逻辑推理，分析以下中文文本的情感类别。\n选项：0:愤怒, 1:悲伤, 2:恐惧, 3:中性, 4:高兴, 5:惊奇"
        examples = """
参考示例：
文本： "谁能告诉我怎么把flyme账户这个SB关了"
推理过程： 文本中直接使用了侮辱性词汇“SB”，且表达了对系统功能的强烈厌烦和无法关闭的挫败感，情绪极度愤怒。
最终答案： 0

文本： "明明想做你最终的伴侣却又无能为力……"
推理过程： 文本表达了极度的无奈、遗憾和爱而不得的深深失落感，属于典型的悲伤情绪。
最终答案： 1

文本： "发现里面有个人，定睛一看，原来是我妹妹，吓得我一愣一愣的，到现在还心有余悸！"
推理过程： 文本描述了突如其来的惊吓，并使用了“吓得我一愣一愣的”和“心有余悸”等词汇，明确表达了恐惧感。
最终答案： 2

文本： "那份内部报告一经泄露，引发了一场前所未有的公共信任危机。"
推理过程： 这是一句纯粹的新闻事实描述，叙述了事件的发展及其影响，没有任何情感词或主观偏好，属于中性陈述。
最终答案： 3

文本： "紧张那么些天终于过了第一次窄角停车了，好开心加油"
推理过程： 文本描述了完成困难任务后的喜悦，并直截了当地使用了“好开心”来表达高兴的情绪。
最终答案： 4

文本： "想不到！河海大学图书馆借阅次数最高的居然是····高数！"
推理过程： 文本开头使用了“想不到！”，结尾使用了感叹号，表达了对这一出人意料事实的强烈惊讶。
最终答案： 5
"""
        target_block = f"现在，请对以下文本进行分类：\n文本： \"{text}\"\n推理过程："

    elif dataset_name == "SST-5":
        task_desc = "Task: Classify the sentiment of the text using step-by-step reasoning.\nOptions: 0:Very Negative, 1:Negative, 2:Neutral, 3:Positive, 4:Very Positive"
        examples = """
Examples:
Text: "an ugly , pointless , stupid movie ."
Reasoning: The reviewer uses three intensely derogatory adjectives—"ugly," "pointless," and "stupid"—leaving no room for redeeming qualities. This is a very negative review.
Final Answer: 0

Text: "... the efforts of its star , kline , to lend some dignity to a dumb story are for naught ."
Reasoning: While it acknowledges the star's effort, the reviewer describes the story as "dumb" and says the efforts were "for naught" (useless), which is a clear negative critique.
Final Answer: 1

Text: "avary 's film never quite emerges from the shadow of ellis ' book ."
Reasoning: The statement compares the film to the original book and finds it somewhat lacking, but it isn't an outright condemnation. It represents a mild, balanced, or neutral critical observation.
Final Answer: 2

Text: "the film does give a pretty good overall picture of the situation in laramie following the murder of matthew shepard ."
Reasoning: The phrase "pretty good overall picture" is a modest endorsement, indicating the film was successful in its goal. This is a positive sentiment.
Final Answer: 3

Text: "... a powerful sequel and one of the best films of the year ."
Reasoning: The phrase "one of the best films of the year" is the highest possible praise for a movie, indicating an extremely positive sentiment.
Final Answer: 4
"""
        target_block = f"Please classify the following text:\nText: \"{text}\"\nReasoning:"

    elif dataset_name == "TweetEval":
        task_desc = "Task: Classify tweet sentiment using step-by-step reasoning.\nOptions: 0:Negative, 1:Neutral, 2:Positive"
        examples = """
Examples:
Text: "Everton Vs. Chelsea so early on a Saturday. These boys want to ruin my weekend. Why Lord."
Reasoning: The user is complaining about the scheduling and claims it will "ruin my weekend," expressing clear frustration and negative sentiment.
Final Answer: 0

Text: "JR’s HAVE to go to the pep rally tomorrow!"
Reasoning: This is a factual announcement or instruction about a school event. It contains no subjective emotional words or evaluation.
Final Answer: 1

Text: "Foo Fighters tomorrow night in Milton Keynes also supported by Royal Blood, can't wait, going to be awesome!"
Reasoning: The phrases "can't wait" and "going to be awesome" express strong positive anticipation and excitement.
Final Answer: 2
"""
        target_block = f"Please classify the following text:\nText: \"{text}\"\nReasoning:"

    if method == "0-Shot CoT": return f"{task_desc}\n\n{target_block}"
    else: return f"{task_desc}\n{examples}\n{target_block}"

# --- 5. 双语解析器 ---
def parse_prediction(response, dataset_name):
    mapping_info = LABEL_MAPPING[dataset_name]
    match = re.search(r'(?:Final Answer|最终答案)[:：]\s*(\d)', response, re.IGNORECASE)
    if match: pred = int(match.group(1))
    else:
        matches = re.findall(r'\d', response)
        pred = int(matches[-1]) if matches else mapping_info["fallback"]
    if pred < 0 or pred > mapping_info["max_id"]: return mapping_info["fallback"]
    return pred

# --- 6. 核心推理函数 (带性能监控) ---
def run_benchmark(dataset_name, method):
    print(f"\n🚀 {dataset_name} | {method} ...")
    val_df = get_validation_set(dataset_name)
    
    preds = []
    responses = [] 
    labels = val_df['label'].tolist()
    
    # 清理显存
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    
    sys_msg = "你是严谨的情感分析专家。请先输出逻辑推理，再输出最终答案。" if dataset_name == "SMP2020" else "You are a logical sentiment analysis expert. Provide reasoning first."

    start_time = time.time()

    for text in tqdm(val_df['text']):
        content = build_cot_prompt(dataset_name, text, method)
        messages = [{"role": "system", "content": sys_msg}, {"role": "user", "content": content}]
        text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer([text_input], return_tensors="pt").to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs, max_new_tokens=150, do_sample=False, temperature=0.0,
                pad_token_id=tokenizer.pad_token_id
            )
        
        response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
        preds.append(parse_prediction(response, dataset_name))
        responses.append(response)
        
    end_time = time.time()
    
    # 物理性能计算
    total_time_sec = end_time - start_time
    avg_latency_ms = (total_time_sec / len(val_df)) * 1000 # 换算成毫秒
    peak_memory_mb = torch.cuda.max_memory_allocated() / (1024 * 1024)
    
    # 学术指标计算
    f1 = f1_score(labels, preds, average="macro")
    acc = accuracy_score(labels, preds)
    
    # 保存具体 Cases
    cases_df = pd.DataFrame({
        "text": val_df['text'],
        "true_label": labels,
        "pred_label": preds,
        "llm_reasoning": responses
    })
    safe_method_name = method.replace(" ", "_")
    safe_model_name = CURRENT_MODEL.split('/')[-1]
    cases_filename = f"{safe_model_name}_{dataset_name}_{safe_method_name}_cases.csv"
    cases_df.to_csv(cases_filename, index=False)
    
    return f1, acc, avg_latency_ms, peak_memory_mb

# --- 7. 启动实验 ---
results = []
datasets = ["SMP2020", "SST-5", "TweetEval"]
methods = ["0-Shot CoT", "Balanced 1-Shot CoT"]

print(f"{'='*80}\n🏆 启动极致公平评估协议 (完全对齐版): {CURRENT_MODEL}\n{'='*80}")
for ds in datasets:
    for method in methods:
        f1, acc, latency, memory = run_benchmark(ds, method)
        print(f"✅ {ds} ({method}) -> F1: {f1:.4f}, Acc: {acc:.4f}, Latency: {latency:.2f}ms, VRAM: {memory:.2f}MB")
        
        results.append({
            "Dataset": ds, 
            "Model": f"{CURRENT_MODEL.split('/')[-1]}", 
            "Method": method,
            "Macro-F1": f1, 
            "Accuracy": acc,
            "Inference_Time_ms": latency,
            "Peak_Memory_MB": memory,
            "Params_M": MODEL_PARAMS_M
        })

final_df = pd.DataFrame(results)
print("\n" + "="*80 + "\n📊 最终评测报表 (全指标)\n" + "="*80)
print(final_df.to_string(index=False))
final_df.to_csv(f"{CURRENT_MODEL.split('/')[-1]}_aligned_cot_results.csv", index=False)

🤖 正在初始化模型: Qwen/Qwen2.5-7B-Instruct ...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

  0%|          | 0/150 [00:00<?, ?it/s]

✅ TweetEval (0-Shot CoT) -> F1: 0.2702, Acc: 0.3400, Latency: 11472.60ms, VRAM: 5614.77MB

🚀 TweetEval | Balanced 1-Shot CoT ...


  0%|          | 0/150 [00:00<?, ?it/s]

✅ TweetEval (Balanced 1-Shot CoT) -> F1: 0.5439, Acc: 0.5400, Latency: 10439.19ms, VRAM: 5639.55MB

📊 最终评测报表 (全指标)
  Dataset               Model              Method  Macro-F1  Accuracy  Inference_Time_ms  Peak_Memory_MB    Params_M
  SMP2020 Qwen2.5-7B-Instruct          0-Shot CoT  0.366608  0.383333       10904.704456     5626.213867 4352.972288
  SMP2020 Qwen2.5-7B-Instruct Balanced 1-Shot CoT  0.608536  0.610000        7183.455234     5680.214355 4352.972288
    SST-5 Qwen2.5-7B-Instruct          0-Shot CoT  0.170038  0.217500       11598.404477     5620.231445 4352.972288
    SST-5 Qwen2.5-7B-Instruct Balanced 1-Shot CoT  0.436023  0.435000       10915.523761     5670.977051 4352.972288
TweetEval Qwen2.5-7B-Instruct          0-Shot CoT  0.270171  0.340000       11472.599413     5614.765625 4352.972288
TweetEval Qwen2.5-7B-Instruct Balanced 1-Shot CoT  0.543931  0.540000       10439.191292     5639.547363 4352.972288
